# 🚀 LCEL 标准组件

LCEL 的强大之处在于其丰富的**标准组件**。所有组件都继承自 **Runnable 接口**，因此都支持以下方法：

- `.invoke()` - 同步调用
- `.stream()` - 流式输出
- `.batch()` - 批量处理
- `.ainvoke()` - 异步调用

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_community.llms import Tongyi

# 获取环境中KEY
tongyi_key = os.environ.get('QWEN_KEY')
# 设置KEY
os.environ["DASHSCOPE_API_KEY"] = tongyi_key
# 创建模型
llm = Tongyi()

---

## 📦 RunnablePassthrough

**作用**：原样传递输入，不做任何修改。

**使用场景**：构建复杂链式结构时作为占位符、分支点或数据透传节点，通常与 `.assign()` 配合使用。

---

### 💡 生活化比喻：点外卖

假设你点了一份 **宫保鸡丁 + 米饭** 的套餐。

| 角色 | 对应组件 | 职责 |
|------|---------|------|
| 👨‍🍳 厨师 | LLM | 只负责做菜 |
| 📋 订单信息 | 原始输入 | 包含订单号、备注"不要辣" |
| 📦 打包员 | RunnablePassthrough | 保留原始订单信息 |

**关键问题**：如果厨师做完菜就把原始订单扔了，那打包时就不知道你的备注和订单号了！

**解决方案**：`RunnablePassthrough` 就是那个"保留原始订单"的人 ✅

In [1]:
from langchain_core.runnables import RunnablePassthrough

pass_through = RunnablePassthrough()
rs = pass_through.invoke("hello")
print(rs)

hello


In [3]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("回答这个问题：{question}")

# ❌ 错误方式：只传 question 给 prompt，原始输入丢了
chain = prompt | llm
result = chain.invoke({"question": "你好吗？"})
print(result)

你好！😊 很高兴你来打招呼～  
作为AI助手，我没有真实的情绪或身体状态，但我始终在线、专注、乐于为你提供帮助！无论是解答问题、一起思考、学习新知识，还是陪你聊聊天、写文字、理思路……我都很乐意～  
你今天过得怎么样？有什么想聊的、想问的，或者需要帮忙的吗？🌟


In [ ]:
# ✅ 正确方式：使用 RunnablePassthrough 保留原始数据
chain = (
    RunnablePassthrough() | 
    (lambda x: {
        'answer': (prompt | llm).invoke(x),
        'original_question': x['question']
    })
)
rs = chain.invoke({'question': '你好吗？'})
print(rs)

{'answer': '你好！😊 很高兴你来打招呼～  \n作为AI助手，我没有真实的情绪或身体状态，但我始终在线、专注、乐于为你提供帮助！无论是解答问题、一起思考、学习新知识，还是陪你聊聊天、写文字、理思路……我都很乐意～  \n你今天过得怎么样？有什么想聊的、想问的，或者需要帮忙的吗？🌟', 'original_question': '你好吗？'}


In [5]:
# ✅ 更优雅的方式：使用 .assign() 方法
chain = (
    RunnablePassthrough()
    .assign(answer=lambda x: (prompt | llm).invoke(x))
)
rs = chain.invoke({'question': '你好吗？'})
print(rs)

{'question': '你好吗？', 'answer': '你好！😊 很高兴你来打招呼～  \n作为AI助手，我没有真实的情绪或身体状态，但我始终在线、专注、乐于为你提供帮助！无论是解答问题、一起思考、学习新知识，还是陪你聊聊生活、写点文字、理清思路……我都很乐意。  \n你今天过得怎么样？有什么想聊的、需要帮忙的，随时告诉我吧！ 🌟'}


---

## 🔧 RunnableLambda

**作用**：将任意 Python 函数（或 lambda 表达式）包装成符合 LCEL 规范的 Runnable 对象。

**核心价值**：让自定义逻辑无缝融入 LangChain 的链式流水线中，支持 `.invoke()`、`.batch()`、`.stream()` 等标准接口。

**使用场景**：
- 数据清洗（去除空格、格式转换）
- 条件判断（路由逻辑）
- 结果后处理（格式化、过滤）
- 与其他 Runnable 通过 `|` 运算符组合

In [6]:
from langchain_core.runnables import RunnableLambda

def add_one(x: int) -> int:
    return x + 1

runnable_func = RunnableLambda(add_one)
rs = runnable_func.invoke(10)
print(rs)

11


In [7]:
# 使用 lambda 表达式
runnable_func = RunnableLambda(lambda x: x * 2)
rs = runnable_func.invoke(6)
print(rs)

12


In [9]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("回答：{text}")

# 创建一个支持链式调用的函数，清空两边空格
clean_input = RunnableLambda(lambda x: x.strip().lower())

# 构建完整链
chain = clean_input | prompt | llm

rs = chain.invoke("   1+1   ")
print(rs)

1 + 1 = 2


---

## 🗺️ RunnableMap

**作用**：并行处理输入字典的多个字段，将不同部分分别送入不同的 Runnable 子组件，然后合并结果。

**核心理念**："把输入拆开，各干各的，再合起来"

**使用场景**：
- 同时处理多个上下文来源（用户问题 + 检索文档 + 用户画像）
- 并行执行多个独立任务，提升效率
- 构建复杂的 LCEL 链

> 💡 **提示**：`RunnableMap` 是 `RunnableParallel` 的别名！

In [11]:
from langchain_core.runnables import RunnableMap, RunnableLambda

# 定义两个功能
upper_case = RunnableLambda(lambda x: x.upper())
length = RunnableLambda(lambda x: len(x))

map_runnable = RunnableMap(
    upper=upper_case,
    length=length
)

rs = map_runnable.invoke("hello")
print(rs)

{'upper': 'HELLO', 'length': 5}


In [12]:
# 复杂示例：同时处理多个字段
map_runnable = RunnableMap(
    format_input=RunnableLambda(lambda x: f"{x['question'].strip()}?"),
    profile=RunnableLambda(lambda x: f'Profile of {x["user_id"]}'),
)
rs = map_runnable.invoke({"question": "你好吗   ", "user_id": "Alice"})
print(rs)

{'format_input': '你好吗?', 'profile': 'Profile of Alice'}


---

## 🌿 RunnableBranch

**作用**：根据输入内容动态选择不同的处理分支，类似于编程中的 `if-elif-else` 逻辑。

**核心理念**："根据输入走不同路"

**使用场景**：
- 根据用户意图执行不同逻辑（问候 vs 计算 vs 普通问答）
- 根据数据类型选择处理方式
- 根据上下文状态切换策略
- 构建智能、自适应的 AI 应用

**工作流程**：
```
输入 → 条件1? → 是 → 执行分支1
     ↓ 否
     条件2? → 是 → 执行分支2
     ↓ 否
     默认分支
```

In [4]:
from langchain_core.runnables import RunnableBranch, RunnableLambda

def is_short_question(x: str) -> bool:
    return len(x) < 5

def is_long_question(x: str) -> bool:
    return len(x) > 10

# 定义不同回答策略
short_answer = RunnableLambda(lambda x: f"简短回答：{x}")
long_answer = RunnableLambda(lambda x: f"详细分析：{x}...")
default_answer = RunnableLambda(lambda x: f"普通回答：{x}")

# 构建分支
branch = RunnableBranch(
    (is_short_question, short_answer),
    (is_long_question, long_answer),
    default_answer  # 默认分支
)

print(branch.invoke("你好?"))              # 简短回答
print(branch.invoke("聊一聊关于langchain的使用"))  # 详细分析
print(branch.invoke("天气怎么样?"))         # 普通回答

简短回答：你好?
详细分析：聊一聊关于langchain的使用...
普通回答：天气怎么样?


In [5]:
# 复杂分支：处理不同类型的问题
def is_math_query(x: dict) -> bool:
    return "计算" in x["question"] or any(op in x["question"] for op in ["+", "-", "*", "/"])

def is_greeting(x: dict) -> bool:
    return any(g in x["question"] for g in ["你好", "hi", "hello"])

math_handler = RunnableLambda(lambda x: {"answer": "结果是 42"})
greeting_handler = RunnableLambda(lambda x: {"answer": "你好！很高兴见到你！"})
fallback_handler = RunnableLambda(lambda x: {"answer": "我不太明白，请换个问法？"})

branch = RunnableBranch(
    (is_math_query, math_handler),
    (is_greeting, greeting_handler),
    fallback_handler
)

result = branch.invoke({"question": "1+1等于几？"})
print(result)  # {'answer': '结果是 42'}

{'answer': '结果是 42'}


In [ ]:
# 高级用法：结合 Prompt 和 LLM
from langchain_core.prompts import ChatPromptTemplate

# 定义不同 prompt 策略
casual_prompt = ChatPromptTemplate.from_template("轻松地回答：{input}")
formal_prompt = ChatPromptTemplate.from_template("正式地回答：{input}")

def is_casual(x: str) -> bool:
    return any(word in x.lower() for word in ["嘿", "嗨", "咋样"])

branch = RunnableBranch(
    (is_casual, casual_prompt | llm),
    formal_prompt | llm  # 默认分支
)

response = branch.invoke("嘿，今天过得怎么样？")
# 使用 casual_prompt
print(response)

嘿～今天过得超棒的！☀️  
和你聊上天，心情就像喝了一杯温热的蜂蜜柠檬水——暖暖的、甜甜的，还带点小清爽～  
你呢？今天有没有遇到什么开心的小事，或者想吐槽的“小乌云”？我随时在线，认真听、笑着接，绝不转台 🌈😄
